In [1]:
import pandas as pd
import os
import numpy as np

In [3]:
def load_screen(screen_name, sl_pair, base_dir="input_data/2_outputs_with_hgnc"):
    """
    Load a screen CSV given only the screen name and SL pair.
    Auto-builds the file path: {base_dir}/{screen}_{sl_pair}_screen_with_hgnc.csv
    """
    filename = f"{screen_name}_{sl_pair}_screen_with_hgnc.csv"
    filepath = os.path.join(base_dir, filename)
    
    df = pd.read_csv(filepath)
    df["Screen"] = screen_name.capitalize()
    df["SL_Pair"] = sl_pair
    return df

# Example usage:
dunn_pten_pik3cb = load_screen("dunn", "PTEN_PIK3CB")
dunn_pten_akt    = load_screen("dunn", "PTEN_AKT")
hayes_nras_cdk = load_screen("hayes", "NRAS_CDK4_6")
krall_braf_mek = load_screen("krall", "BRAF_MEK")



In [5]:
def load_aggregated_with_meta(screen_name, sl_pair, base_dir="input_data/2_outputs_with_hgnc"):
    """
    Load an aggregated screen CSV.
    File format: {sl_pair}_{screen}_aggregated.csv
    """
    filename = f"{sl_pair}_{screen_name}_aggregated.csv"
    filepath = os.path.join(base_dir, filename)
    
    df = pd.read_csv(filepath)
    df["Screen"] = screen_name
    df["SL_Pair"] = sl_pair
    return df

# Example usage:
awwad_llorca_arid1a_atr = load_aggregated_with_meta("Awwad_Llorca", "ARID1A_ATR")
krall_wang_yu_kras_mek = load_aggregated_with_meta("Krall_Wang_Yu","KRAS_MEK")
noordermeer_zimmermann_brca1_parp1 = load_aggregated_with_meta("Noordermeer_Zimmermann","BRCA1_PARP1")
hayes_kral_nras_mek = load_aggregated_with_meta("Hayes_Krall","NRAS_MEK")

In [6]:
df_combined = pd.concat([
    dunn_pten_pik3cb, dunn_pten_akt,
    hayes_nras_cdk, krall_braf_mek,
    awwad_llorca_arid1a_atr,
    krall_wang_yu_kras_mek,
    noordermeer_zimmermann_brca1_parp1,
    hayes_kral_nras_mek
], ignore_index=True)
df_combined

,Cell_line,Resistance,Gene,Gene_corrected,hgnc_id,ensembl_gene_id,entrez_id,Screen,SL_Pair
0,3 cell line,Resistant,PIK3R2,PIK3R2,HGNC:8980,ENSG00000105647,5296.0,Dunn,PTEN_PIK3CB
1,NaN,Resistant,INPPL1,INPPL1,HGNC:6080,ENSG00000165458,3636.0,Dunn,PTEN_PIK3CB
2,NaN,Resistant,TSC2,TSC2,HGNC:12363,ENSG00000103197,7249.0,Dunn,PTEN_PIK3CB
3,2 cell line,Resistant,NF2,NF2,HGNC:7773,ENSG00000186575,4771.0,Dunn,PTEN_PIK3CB
4,NaN,Resistant,CXorf56,CXorf56,HGNC:26239,ENSG00000018610,63932.0,Dunn,PTEN_PIK3CB
...,...,...,...,...,...,...,...,...,...
144075,NaN,Not_Resistant,ZYG11A,ZYG11A,HGNC:32058,ENSG00000203995,440590.0,Hayes_Krall,NRAS_MEK
144076,NaN,Not_Resistant,ZYG11B,ZYG11B,HGNC:25820,ENSG00000162378,79699.0,Hayes_Krall,NRAS_MEK
144077,NaN,Not_Resistant,ZYX,ZYX,HGNC:13200,ENSG00000159840,7791.0,Hayes_Krall,NRAS_MEK
144078,NaN,Not_Resistant,ZZEF1,ZZEF1,HGNC:29027,ENSG00000074755,23140.0,Hayes_Krall,NRAS_MEK


In [7]:
def split_sl_pair(sl_pair):
    parts = sl_pair.split('_')
    
    # Special case: any MEK-containing pair
    if "MEK" in parts:
        return parts[0], "MEK1", "MEK2"
    
    # Special case: CDK4_6
    if len(parts) == 3 and parts[1] == "CDK4" and parts[2] == "6":
        return parts[0], "CDK4", "CDK6"
    
    # Standard case: 2 genes
    if len(parts) == 2:
        return parts[0], parts[1], None
    
    # Standard case: 3 parts (non-MEK/CDK special handling)
    if len(parts) == 3:
        return parts[0], parts[1], parts[2]
    
    # Fallback
    return parts[0], None, None

df_combined[['Biomarker', 'Target1', 'Target2']] = df_combined['SL_Pair'].apply(lambda x: pd.Series(split_sl_pair(x)))
df_combined

,Cell_line,Resistance,Gene,Gene_corrected,hgnc_id,ensembl_gene_id,entrez_id,Screen,SL_Pair,Biomarker,Target1,Target2
0,3 cell line,Resistant,PIK3R2,PIK3R2,HGNC:8980,ENSG00000105647,5296.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None
1,NaN,Resistant,INPPL1,INPPL1,HGNC:6080,ENSG00000165458,3636.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None
2,NaN,Resistant,TSC2,TSC2,HGNC:12363,ENSG00000103197,7249.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None
3,2 cell line,Resistant,NF2,NF2,HGNC:7773,ENSG00000186575,4771.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None
4,NaN,Resistant,CXorf56,CXorf56,HGNC:26239,ENSG00000018610,63932.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None
...,...,...,...,...,...,...,...,...,...,...,...,...
144075,NaN,Not_Resistant,ZYG11A,ZYG11A,HGNC:32058,ENSG00000203995,440590.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2
144076,NaN,Not_Resistant,ZYG11B,ZYG11B,HGNC:25820,ENSG00000162378,79699.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2
144077,NaN,Not_Resistant,ZYX,ZYX,HGNC:13200,ENSG00000159840,7791.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2
144078,NaN,Not_Resistant,ZZEF1,ZZEF1,HGNC:29027,ENSG00000074755,23140.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2


In [8]:
hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')
# Build lookup dictionary
gene_lookup = {}

# Helper function to normalize symbols
def normalize_symbol(s):
    return s.strip().upper() if isinstance(s, str) else None

for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)
    entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'

    # Primary symbol
    symbol = normalize_symbol(entry.get('symbol'))
    if symbol:
        gene_lookup[symbol] = (hgnc_id, ensembl_id, entrez_id)

    # Previous symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('prev_symbol')):
        prev_symbols = entry['prev_symbol'].split('|')
        for prev_symbol in prev_symbols:
            norm_prev = normalize_symbol(prev_symbol)
            if norm_prev and norm_prev not in gene_lookup:
                gene_lookup[norm_prev] = (hgnc_id, ensembl_id, entrez_id)

    # Alias symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('alias_symbol')):
        alias_symbols = entry['alias_symbol'].split('|')
        for alias in alias_symbols:
            norm_alias = normalize_symbol(alias)
            if norm_alias and norm_alias not in gene_lookup:
                gene_lookup[norm_alias] = (hgnc_id, ensembl_id, entrez_id)

# Lookup function using normalized input
def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

# Apply lookup to relevant columns in df_combined
df_combined['hgnc_id_biomarker'], df_combined['ensembl_gene_id_biomarker'], df_combined['entrez_id_biomarker'] = zip(
    *df_combined['Biomarker'].apply(lookup_gene_info)
)
df_combined['hgnc_id_target1'], df_combined['ensembl_gene_id_target1'], df_combined['entrez_id_target1'] = zip(
    *df_combined['Target1'].apply(lookup_gene_info)
)
df_combined['hgnc_id_target2'], df_combined['ensembl_gene_id_target2'], df_combined['entrez_id_target2'] = zip(
    *df_combined['Target2'].apply(lookup_gene_info)
)

# Force entrez_id columns to string type
df_combined['entrez_id_biomarker'] = df_combined['entrez_id_biomarker'].astype(str)
df_combined['entrez_id_target1'] = df_combined['entrez_id_target1'].astype(str)
df_combined['entrez_id_target2'] = df_combined['entrez_id_target2'].astype(str)
df_combined

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_79773/511634035.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')


,Cell_line,Resistance,Gene,Gene_corrected,hgnc_id,ensembl_gene_id,entrez_id,Screen,SL_Pair,Biomarker,...,Target2,hgnc_id_biomarker,ensembl_gene_id_biomarker,entrez_id_biomarker,hgnc_id_target1,ensembl_gene_id_target1,entrez_id_target1,hgnc_id_target2,ensembl_gene_id_target2,entrez_id_target2
0,3 cell line,Resistant,PIK3R2,PIK3R2,HGNC:8980,ENSG00000105647,5296.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
1,NaN,Resistant,INPPL1,INPPL1,HGNC:6080,ENSG00000165458,3636.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
2,NaN,Resistant,TSC2,TSC2,HGNC:12363,ENSG00000103197,7249.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
3,2 cell line,Resistant,NF2,NF2,HGNC:7773,ENSG00000186575,4771.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
4,NaN,Resistant,CXorf56,CXorf56,HGNC:26239,ENSG00000018610,63932.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,NaN,Not_Resistant,ZYG11A,ZYG11A,HGNC:32058,ENSG00000203995,440590.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144076,NaN,Not_Resistant,ZYG11B,ZYG11B,HGNC:25820,ENSG00000162378,79699.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144077,NaN,Not_Resistant,ZYX,ZYX,HGNC:13200,ENSG00000159840,7791.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144078,NaN,Not_Resistant,ZZEF1,ZZEF1,HGNC:29027,ENSG00000074755,23140.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0


In [11]:
df_combined['Resistance'] = df_combined['Resistance'].replace({"NotResistant": "Non_Resistance"})
df_combined['Resistance'] = df_combined['Resistance'].replace({"Not_Resistant": "Non_Resistance"})
df_combined['Resistance'] = df_combined['Resistance'].replace({"Resistant": "Resistance"})

df_combined


,Cell_line,Resistance,Gene,Gene_corrected,hgnc_id,ensembl_gene_id,entrez_id,Screen,SL_Pair,Biomarker,...,Target2,hgnc_id_biomarker,ensembl_gene_id_biomarker,entrez_id_biomarker,hgnc_id_target1,ensembl_gene_id_target1,entrez_id_target1,hgnc_id_target2,ensembl_gene_id_target2,entrez_id_target2
0,3 cell line,Resistance,PIK3R2,PIK3R2,HGNC:8980,ENSG00000105647,5296.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
1,NaN,Resistance,INPPL1,INPPL1,HGNC:6080,ENSG00000165458,3636.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
2,NaN,Resistance,TSC2,TSC2,HGNC:12363,ENSG00000103197,7249.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
3,2 cell line,Resistance,NF2,NF2,HGNC:7773,ENSG00000186575,4771.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
4,NaN,Resistance,CXorf56,CXorf56,HGNC:26239,ENSG00000018610,63932.0,Dunn,PTEN_PIK3CB,PTEN,...,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,NaN,Non_Resistance,ZYG11A,ZYG11A,HGNC:32058,ENSG00000203995,440590.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144076,NaN,Non_Resistance,ZYG11B,ZYG11B,HGNC:25820,ENSG00000162378,79699.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144077,NaN,Non_Resistance,ZYX,ZYX,HGNC:13200,ENSG00000159840,7791.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144078,NaN,Non_Resistance,ZZEF1,ZZEF1,HGNC:29027,ENSG00000074755,23140.0,Hayes_Krall,NRAS_MEK,NRAS,...,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0


In [12]:
df_combined['Resistance'].unique()

array(['Resistance', 'Non_Resistance'], dtype=object)

In [13]:
df_combined = df_combined.rename(columns={
    "Resistance": "Class",
    "Gene_corrected": "Query",
    "hgnc_id": "hgnc_id_query",
    "ensembl_gene_id": "ensembl_gene_id_query",
    "entrez_id": "entrez_id_query",
    
})

df_combined = df_combined.drop(columns=["Cell_line","Gene"])
df_combined

,Class,Query,hgnc_id_query,ensembl_gene_id_query,entrez_id_query,Screen,SL_Pair,Biomarker,Target1,Target2,hgnc_id_biomarker,ensembl_gene_id_biomarker,entrez_id_biomarker,hgnc_id_target1,ensembl_gene_id_target1,entrez_id_target1,hgnc_id_target2,ensembl_gene_id_target2,entrez_id_target2
0,Resistance,PIK3R2,HGNC:8980,ENSG00000105647,5296.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
1,Resistance,INPPL1,HGNC:6080,ENSG00000165458,3636.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
2,Resistance,TSC2,HGNC:12363,ENSG00000103197,7249.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
3,Resistance,NF2,HGNC:7773,ENSG00000186575,4771.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
4,Resistance,CXorf56,HGNC:26239,ENSG00000018610,63932.0,Dunn,PTEN_PIK3CB,PTEN,PIK3CB,None,HGNC:9588,ENSG00000171862,5728.0,HGNC:8976,ENSG00000051382,5291.0,NaN,NaN,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Non_Resistance,ZYG11A,HGNC:32058,ENSG00000203995,440590.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144076,Non_Resistance,ZYG11B,HGNC:25820,ENSG00000162378,79699.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144077,Non_Resistance,ZYX,HGNC:13200,ENSG00000159840,7791.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0
144078,Non_Resistance,ZZEF1,HGNC:29027,ENSG00000074755,23140.0,Hayes_Krall,NRAS_MEK,NRAS,MEK1,MEK2,HGNC:7989,ENSG00000213281,4893.0,HGNC:6840,ENSG00000169032,5604.0,HGNC:6842,ENSG00000126934,5605.0


In [14]:
column_order = (
    ["Screen", "SL_Pair"] +  # Move "Screen" and "SL_Pair" to the beginning
    ["Query",'Biomarker','Target1','Target2', "entrez_id_query","hgnc_id_query","ensembl_gene_id_query",
    "entrez_id_biomarker","hgnc_id_biomarker", "ensembl_gene_id_biomarker","entrez_id_target1","hgnc_id_target1",
    "ensembl_gene_id_target1","entrez_id_target2","hgnc_id_target2","ensembl_gene_id_target2"] +  # Keep "Query" and "ensembl_gene_id" together
    [col for col in df_combined.columns if col not in ["Screen", "SL_Pair", "Query", "entrez_id_query","hgnc_id_query","ensembl_gene_id_query","Biomarker", "Target1", 
                                                     "Target2","entrez_id_biomarker","hgnc_id_biomarker", "ensembl_gene_id_biomarker","entrez_id_target1",
                                                     "hgnc_id_target1","ensembl_gene_id_target1","entrez_id_target2","hgnc_id_target2","ensembl_gene_id_target2"]])  # Remaining columns
df_combined =df_combined[column_order]  # Reorder dataframe
df_combined

,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,hgnc_id_biomarker,ensembl_gene_id_biomarker,entrez_id_target1,hgnc_id_target1,ensembl_gene_id_target1,entrez_id_target2,hgnc_id_target2,ensembl_gene_id_target2,Class
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,None,5296.0,HGNC:8980,ENSG00000105647,5728.0,HGNC:9588,ENSG00000171862,5291.0,HGNC:8976,ENSG00000051382,NA,NaN,NaN,Resistance
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,None,3636.0,HGNC:6080,ENSG00000165458,5728.0,HGNC:9588,ENSG00000171862,5291.0,HGNC:8976,ENSG00000051382,NA,NaN,NaN,Resistance
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,None,7249.0,HGNC:12363,ENSG00000103197,5728.0,HGNC:9588,ENSG00000171862,5291.0,HGNC:8976,ENSG00000051382,NA,NaN,NaN,Resistance
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,None,4771.0,HGNC:7773,ENSG00000186575,5728.0,HGNC:9588,ENSG00000171862,5291.0,HGNC:8976,ENSG00000051382,NA,NaN,NaN,Resistance
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,None,63932.0,HGNC:26239,ENSG00000018610,5728.0,HGNC:9588,ENSG00000171862,5291.0,HGNC:8976,ENSG00000051382,NA,NaN,NaN,Resistance
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,HGNC:7989,ENSG00000213281,5604.0,HGNC:6840,ENSG00000169032,5605.0,HGNC:6842,ENSG00000126934,Non_Resistance
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,HGNC:7989,ENSG00000213281,5604.0,HGNC:6840,ENSG00000169032,5605.0,HGNC:6842,ENSG00000126934,Non_Resistance
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,HGNC:7989,ENSG00000213281,5604.0,HGNC:6840,ENSG00000169032,5605.0,HGNC:6842,ENSG00000126934,Non_Resistance
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,HGNC:7989,ENSG00000213281,5604.0,HGNC:6840,ENSG00000169032,5605.0,HGNC:6842,ENSG00000126934,Non_Resistance


In [15]:
df_combined[["StringInteractionWithBiomarker", "StringInteractionWithTarget", "CoexpressionWithBiomarker",
             "CoexpressionWithTarget","AvgExpression","CoessentialityWithBiomarker",
             "CoessentialityWithTarget","FET_SharedInteractors_Biomarker_BIOGRID","FET_SharedInteractors_Target_BIOGRID","FET_SharedInteractors_Biomarker_STRING","FET_SharedInteractors_Target_STRING"]] = None
df_combined.to_csv("input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv", index=False)


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_79773/2245159733.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_combined[["StringInteractionWithBiomarker", "StringInteractionWithTarget", "CoexpressionWithBiomarker",
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_79773/2245159733.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_combined[["StringInteractionWithBiomarker", "StringInteractionWithTarget", "CoexpressionWithBiomarker",
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh00